In [0]:
"""NeuroPlex gnomAD Ingestion Task.

Scheduled task for Lakeflow Job. Fetches gene constraint + ClinVar variants
for neuroscience target genes and writes to neuroplex_gnomad Delta table.
"""
import sys, time, json, requests
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F

import sys, os
from pathlib import PurePosixPath

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = ctx.notebookPath().get()
p = PurePosixPath(notebook_path)
repo_root = str(p.parent.parent if p.parent.name == "ingestion" else p.parent)
sys.path.insert(0, repo_root)


# Force clean import
for mod in list(sys.modules.keys()):
    if "ingestion" in mod:
        del sys.modules[mod]

from ingestion.source_registry import SOURCE_MAP
from ingestion.ingestors.gnomad import GnomadIngestor

# ── Configuration ──
TARGET_GENES = [
    "HCRT", "HCRTR1", "HCRTR2",
    "PSEN1", "PSEN2", "APP", "MAPT", "GRN", "TARDBP", "C9orf72",
    "SOD1", "FUS", "OPTN", "TBK1",
    "CHD8", "SCN2A", "SYNGAP1", "DYRK1A", "ADNP",
    "CLOCK", "PER2", "CRY1",
    "SLC18A2", "DBH", "TH",
]
from config.neuroplex_config import load_config
CFG = load_config()
TABLE = CFG.prefixed_fqn("gnomad")

# ── Ingest ──
ingestor = GnomadIngestor(SOURCE_MAP["gnomad"])
all_rows = []
errors = []

for gene in TARGET_GENES:
    try:
        for raw in ingestor.fetch(gene=gene, include_variants=False, limit=50):
            all_rows.append(ingestor.normalize(raw).to_row())
    except Exception as e:
        errors.append((gene, str(e)[:80]))

print(f"Fetched {len(all_rows)} records, {len(errors)} errors")

# ── Write ──
schema = StructType([
    StructField("record_id", StringType(), False),
    StructField("source_key", StringType(), False),
    StructField("gene_symbol", StringType(), True),
    StructField("disease", StringType(), True),
    StructField("drug", StringType(), True),
    StructField("title", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("payload", StringType(), False),
    StructField("ingested_at", StringType(), False),
    StructField("source_updated_at", StringType(), True),
])

df = spark.createDataFrame(all_rows, schema=schema)
df = df.withColumn("ingested_at", F.to_timestamp("ingested_at")) \
       .withColumn("source_updated_at", F.to_timestamp("source_updated_at")) \
       .withColumn("payload", F.parse_json("payload"))
df.write.mode("overwrite").saveAsTable(TABLE)

count = spark.sql(f"SELECT COUNT(*) FROM {TABLE}").collect()[0][0]
print(f"\u2705 {TABLE}: {count} records written")
if errors:
    print(f"\u26A0\uFE0F Errors: {errors}")